# Yuda — Heavy Models + Advanced
RTX 4050 6GB | batch_size=32 | accumulation_steps=1

In [ ]:
import sys
sys.path.insert(0, "../..")

from modules.utils.config import *
from modules.utils.seed import set_seed
from modules.data.dataset import get_dataloaders
from modules.models.factory import TrashClassifier
from modules.training.train import fit

set_seed(SEED)
print(f"Device: {DEVICE}")

In [ ]:
MODELS = [
    "resnet50",
    "convnext_tiny",
    "vit_small_patch16_224",
    "efficientnet_b3",
    "resnext50_32x4d",
]

BATCH_SIZE = 32
ACCUMULATION_STEPS = 1
NUM_WORKERS = 4
EPOCHS_HEAD = 10
EPOCHS_FINETUNE = 20
LR_HEAD = 1e-3
LR_FINETUNE = 1e-4
PATIENCE = 10

In [ ]:
train_loader, val_loader, val_ds = get_dataloaders(
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
)
print(f"Train: {len(train_loader.dataset)} | Val: {len(val_loader.dataset)}")

In [ ]:
results = []
for model_name in MODELS:
    model = TrashClassifier(model_name, num_classes=NUM_CLASSES).to(DEVICE)
    res = fit(
        model, train_loader, val_loader,
        name=f"yuda_{model_name}",
        encoder_name=model_name,
        accumulation_steps=ACCUMULATION_STEPS,
        epochs_head=EPOCHS_HEAD,
        epochs_finetune=EPOCHS_FINETUNE,
        lr_head=LR_HEAD,
        lr_finetune=LR_FINETUNE,
        patience=PATIENCE,
        class_weights=val_ds.class_weights,
        device=DEVICE,
    )
    results.append(res)
    print(f">>> {model_name}: val_f1_macro = {res['best_val_f1']:.4f}")

In [ ]:
import pandas as pd
summary = pd.DataFrame(results)
summary = summary.sort_values("best_val_f1", ascending=False)
summary.to_csv(RESULTS / "yuda_summary.csv", index=False)
summary[["name", "best_val_f1", "f1_per_class"]]

## Advanced: Focal Loss + Mixup on best model

In [ ]:
import torch.nn as nn

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss = nn.functional.cross_entropy(inputs, targets, weight=self.alpha, reduction="none")
        pt = (-ce_loss).exp()
        return (pt.clamp(min=1e-8).pow(self.gamma) * ce_loss).mean()


best_model_name = MODELS[0]  # ganti setelah lihat hasil baseline

experiments = [
    ("focal_gamma2", FocalLoss(alpha=val_ds.class_weights.to(DEVICE))),
    ("ce_weighted", None),
]

for loss_name, loss_fn in experiments:
    model = TrashClassifier(best_model_name, num_classes=NUM_CLASSES).to(DEVICE)
    res = fit(
        model, train_loader, val_loader,
        name=f"yuda_adv_{best_model_name}_{loss_name}",
        encoder_name=best_model_name,
        accumulation_steps=ACCUMULATION_STEPS,
        epochs_head=EPOCHS_HEAD,
        epochs_finetune=EPOCHS_FINETUNE,
        lr_head=LR_HEAD,
        lr_finetune=LR_FINETUNE,
        patience=PATIENCE,
        criterion=loss_fn,
        device=DEVICE,
    )
    results.append(res)